In [4]:
import os
from google.genai import client, types
from gitsource import GithubRepositoryDataReader

# Fetch the dataset files
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
files = reader.read()

# Parse files into structural dictionaries
documents = []
for file in files:
    doc = file.parse()
    documents.append(doc)

# Answer for Q1
print(f"Q1: Total lesson pages in dataset = {len(documents)}")

Q1: Total lesson pages in dataset = 72


In [5]:
import minsearch

# Initialize and fit the index for full documents
index = minsearch.Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
index.fit(documents)

# Define our core query
query = "How does the agentic loop keep calling the model until it stops?"

# Execute search
search_results = index.search(
    query=query,
    num_results=5
)

# Answer for Q2
if search_results:
    print(f"Q2: Filename of the first result = {search_results[0]['filename']}")

Q2: Filename of the first result = 01-agentic-rag/lessons/14-agentic-loop.md


In [6]:
# Initialize your Gemini Client (ensure GEMINI_API_KEY is in your environment)
ai_client = client.Client()

INSTRUCTIONS = "You are a course teaching assistant. Answer the user question based on the provided search context."

def build_prompt(query, search_results):
    context = ""
    for doc in search_results:
        context += f"Filename: {doc['filename']}\nContent: {doc['content']}\n\n"
        
    prompt = f"Context:\n{context}\n\nQuestion: {query}"
    return prompt

def rag_with_gemini(query, search_index, num_results=5):
    # 1. Search the index
    results = search_index.search(query=query, num_results=num_results)
    
    # 2. Format the prompt
    prompt = build_prompt(query, results)
    
    # 3. Request completion using system instruction configurations
    config = types.GenerateContentConfig(
        system_instruction=INSTRUCTIONS
    )
    
    response = ai_client.models.generate_content(
        model="gemini-3-flash-preview",
        contents=prompt,
        config=config
    )
    
    # Extract usage statistics
    input_tokens = response.usage_metadata.prompt_token_count
    return response.text, input_tokens

# Run Q3
answer_q3, tokens_q3 = rag_with_gemini(query, index, num_results=5)
print(f"Q3: Input (prompt) tokens sent to model = {tokens_q3}")

Q3: Input (prompt) tokens sent to model = 7923


In [7]:
from gitsource import chunk_documents

# Chunk the original list of document dictionaries
chunks = chunk_documents(documents, size=2000, step=1000)

# Answer for Q4
print(f"Q4: Total chunks generated = {len(chunks)}")

Q4: Total chunks generated = 295


In [8]:
# Initialize and fit the chunk index
chunk_index = minsearch.Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
chunk_index.fit(chunks)

# Run the query against the chunk index
answer_q5, tokens_q5 = rag_with_gemini(query, chunk_index, num_results=5)

print(f"Q5: Input tokens with chunking = {tokens_q5}")
print(f"Token reduction factor: {tokens_q3 / tokens_q5:.1f}x fewer tokens")

Q5: Input tokens with chunking = 2576
Token reduction factor: 3.1x fewer tokens


In [9]:
# Create a local tracking counter variable
search_counter = 0

def course_knowledge_search(search_keyword: str) -> str:
    """
    Searches the course lesson database for information using a text query keyword string.
    
    Args:
        search_keyword: The topic text or keyword query to lookup.
    Returns:
        A compiled string containing the matching context lessons.
    """
    global search_counter
    search_counter += 1  # Increment whenever the agent invokes the tool
    
    results = chunk_index.search(query=search_keyword, num_results=3)
    context = ""
    for doc in results:
        context += f"Source: {doc['filename']}\nContent: {doc['content']}\n\n"
    return context

# Configure Gemini Agent containing the search tool
agent_instructions = (
    "You're a course teaching assistant. Answer the student's question using the "
    "course_knowledge_search tool. Make multiple searches with different keywords before answering."
)

agent_config = types.GenerateContentConfig(
    system_instruction=agent_instructions,
    tools=[course_knowledge_search], # Provide the actual function as a tool asset
)

agent_query = "How does the agentic loop work, and how is it different from plain RAG?"

# Run the agent turn loop execution
response = ai_client.models.generate_content(
    model="gemini-3-flash-preview",
    contents=agent_query,
    config=agent_config
)

# Execute the tool calls if the model requests them
if response.function_calls:
    for call in response.function_calls:
        if call.name == "course_knowledge_search":
            # Extract arguments requested by the model
            args = call.args
            # Execute the function locally
            tool_output = course_knowledge_search(args.get("search_keyword"))
            
            # Send the data back to the model to finalize its response
            final_response = ai_client.models.generate_content(
                model="gemini-3-flash-preview",
                contents=[
                    agent_query,
                    response.candidates[0].content, # History of the call
                    types.Content(
                        role="tool",
                        parts=[types.Part.from_function_response(
                            name="course_knowledge_search",
                            response={"result": tool_output}
                        )]
                    )
                ],
                config=agent_config
            )
            print("\nFinal Agent Answer:\n", final_response.text)

print(f"\nQ6: The search tool was invoked a total of {search_counter} times.")


Q6: The search tool was invoked a total of 4 times.
